# ShotShield v3 — hard negatives

v2 learned a shortcut: in UrbanSound8K, almost the only short, sharp, broadband sound is a gunshot, so a finger snap
next to a laptop microphone scores 97%. v3 keeps the same input format and network (so the browser demo works
unchanged) and changes the **data**:

| | v2 | v3 |
|---|---|---|
| Gunshots | 374 (UrbanSound8K) | + 482 from FSD50K (other recordings, other guns) |
| Other sounds | 8,358 urban sounds (9 classes) | + ESC-50 (2,000 clips, 50 classes) + ~9,000 FSD50K hard negatives: finger snaps, claps, knocks, door slams, dishes, typing, fireworks, fans, microwaves, engines, … |
| Augmentation | none | one extra copy of every training clip with room reverb, microphone EQ, background noise and clipping |
| Duplicates | – | clips whose Freesound ID is in UrbanSound8K are removed from ESC-50/FSD50K (all three come from Freesound) |

**Evaluation stays comparable with v2:** the UrbanSound8K 10-fold cross-validation still tests only on UrbanSound8K.
Two new test sets are added, which no model ever trains on: **hard negatives** (ESC-50 fold 5 + FSD50K eval) and
**out-of-domain gunshots** (FSD50K eval). The released v2 model is scored on exactly the same clips.

**Before you run this notebook on Kaggle**
1. **Add Input** → **UrbanSound8K** (the same dataset as v2).
2. **Settings → Accelerator → GPU T4**, **Settings → Internet → On**. ESC-50 (GitHub, 0.6 GB) and the FSD50K clips
   (a Hugging Face mirror with one file per clip, about 6 GB) are downloaded automatically.
3. Use **Save Version → Save & Run All (Commit)**. With cross-validation it takes about **4–5 hours**, so it must run
   in the background. Set `RUN_CV = False` below for a ~1 hour run (final model only).
4. When it finishes, open the version's **Output** and download **`shotshield_v3_outputs.zip`** (model, results,
   charts) and **`shotshield_v3_web.zip`** (for the browser demo).

In [ ]:
# ONNX conversion tools for the browser demo. tf2onnx is installed without its dependencies:
# it pins an old protobuf that would break TensorFlow.
!pip install -q onnx onnxruntime
!pip install -q --no-deps tf2onnx

In [ ]:
import os, sys, gc, json, glob, time, shutil, hashlib, zipfile, datetime, subprocess, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import scipy.signal
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score, average_precision_score,
                             confusion_matrix, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
import keras
from keras import layers

# ---- Settings -------------------------------------------------------------
SEED = 42
RUN_CV = True          # 10-fold cross-validation (~4 h on a T4). False = final model only (~1 h).
                       # Already done once (results in the README), so False is fine for a re-run.
EPOCHS = 40            # upper bound; early stopping usually ends sooner
BATCH_SIZE = 64
AUGMENT = True         # one augmented copy of every training clip
FSD_CAP = {'dev': 80, 'eval': 30}       # FSD50K clips per hard-negative class
FSD_OTHER = {'dev': 800, 'eval': 300}   # plus this many random other FSD50K clips (speech, music, animals, ...)

INPUT_DIR = '/kaggle/input'
WORK = '/kaggle/working'
DATA = '/tmp/shotshield_data'          # downloads and caches; /tmp is not part of the notebook output
OUT_DIR = f'{WORK}/shotshield_v3'
WEB_DIR = f'{WORK}/shotshield_v3_web'

REPO_RAW = 'https://raw.githubusercontent.com/nishttw/ShotShield/main'
V2_RELEASE = 'https://github.com/nishttw/ShotShield/releases/download/v2.0'
ESC50_ZIP = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
FSD_HF = 'https://huggingface.co/datasets/Fhrozen/FSD50k/resolve/main'

In [ ]:
ASSET_DIR = f'{OUT_DIR}/assets'
for d in (DATA, OUT_DIR, ASSET_DIR, WEB_DIR, f'{WEB_DIR}/examples'):
    os.makedirs(d, exist_ok=True)
keras.utils.set_random_seed(SEED)


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def fetch(url, path, tries=6):
    # Download url to path (skipped if it is already there), retrying with back-off.
    if os.path.exists(path):
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for i in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(path + '.part', 'wb') as f:
                shutil.copyfileobj(r, f, 1 << 20)
            os.replace(path + '.part', path)
            return path
        except urllib.error.HTTPError as e:
            if e.code == 404 or i == tries - 1:
                raise
        except Exception:
            if i == tries - 1:
                raise
        time.sleep(min(60, 2 ** i))


# The exact feature code from the repository (same as v2 and the browser demo)
fetch(f'{REPO_RAW}/src/shotshield/features.py', f'{DATA}/shotshield_features.py')
sys.path.insert(0, DATA)
os.environ['PYTHONPATH'] = DATA + os.pathsep + os.environ.get('PYTHONPATH', '')   # for joblib workers
import shotshield_features as F
CFG = F.FeatureConfig()
SR, N = CFG.sample_rate, CFG.n_samples

print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| librosa', librosa.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Data
### UrbanSound8K
Exactly as in v2: all 8,732 clips, official folds.

In [ ]:
csv_hits, fold1_hits = [], []
for root, dirs, files in os.walk(INPUT_DIR, followlinks=True):
    csv_hits += [os.path.join(root, f) for f in files if f.lower() == 'urbansound8k.csv']
    fold1_hits += [os.path.join(root, d) for d in dirs if d.lower() == 'fold1']
if not csv_hits or not fold1_hits:
    raise FileNotFoundError('UrbanSound8K not found. Click "Add Input", search "UrbanSound8K", add it, then run again.')

meta = pd.read_csv(sorted(csv_hits)[0])
audio_root = os.path.dirname(sorted(fold1_hits)[0])
us8k = pd.DataFrame({
    'source': 'us8k',
    'name': meta['slice_file_name'],
    'path': [os.path.join(audio_root, f'fold{f}', n) for f, n in zip(meta['fold'], meta['slice_file_name'])],
    'fsid': meta['fsID'].astype(int),
    'label': (meta['class'] == 'gun_shot').astype(int),
    'category': meta['class'],
    'group': np.where(meta['class'] == 'gun_shot', 'gunshot', 'urban'),
    'fold': meta['fold'].astype(int),
    'split': 'us8k',
})
print(f'UrbanSound8K: {len(us8k)} clips, {us8k.label.sum()} gunshots')

### ESC-50
2,000 five-second clips in 50 classes: clapping, door knocks, mouse clicks, typing, glass breaking, fireworks,
vacuum cleaners, washing machines, … None is a gunshot, so all are negatives. Its own folds: 1–3 train, 4 validation,
**5 test** (never trained on).

In [ ]:
ESC_IMPULSIVE = {'clapping', 'door_wood_knock', 'mouse_click', 'keyboard_typing', 'can_opening', 'glass_breaking',
                 'footsteps', 'clock_tick', 'fireworks', 'sneezing', 'coughing', 'water_drops', 'thunderstorm',
                 'crackling_fire', 'door_wood_creaks'}
ESC_MACHINE = {'washing_machine', 'vacuum_cleaner', 'engine', 'train', 'airplane', 'helicopter', 'chainsaw',
               'hand_saw', 'toilet_flush', 'clock_alarm'}
# steady sounds used as backgrounds in augmentation (training folds only)
BACKGROUND = ['rain', 'sea_waves', 'wind', 'crickets', 'insects', 'chirping_birds', 'pouring_water', 'crackling_fire',
              'washing_machine', 'vacuum_cleaner', 'engine', 'train', 'airplane', 'helicopter']

esc_csv = [os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR, followlinks=True) for f in fs if f.lower() == 'esc50.csv']
if esc_csv:
    esc_root = os.path.dirname(os.path.dirname(sorted(esc_csv)[0]))
    print('ESC-50 attached:', esc_root)
else:
    zpath = fetch(ESC50_ZIP, f'{DATA}/esc50.zip')
    if not os.path.isdir(f'{DATA}/ESC-50-master'):
        with zipfile.ZipFile(zpath) as z:
            z.extractall(DATA)
    esc_csv = [f'{DATA}/ESC-50-master/meta/esc50.csv']
    esc_root = f'{DATA}/ESC-50-master'
    print('ESC-50 downloaded from GitHub')

esc_meta = pd.read_csv(sorted(esc_csv)[0])
wav_index = {f: os.path.join(r, f) for r, _, fs in os.walk(esc_root, followlinks=True) for f in fs if f.endswith('.wav')}
esc = pd.DataFrame({
    'source': 'esc50',
    'name': esc_meta['filename'],
    'path': esc_meta['filename'].map(wav_index),
    'fsid': esc_meta['src_file'].astype(int),
    'label': 0,
    'category': esc_meta['category'],
    'group': ['impulsive' if c in ESC_IMPULSIVE else 'machine' if c in ESC_MACHINE else 'other' for c in esc_meta['category']],
    'fold': 0,
    'split': esc_meta['fold'].map({1: 'train', 2: 'train', 3: 'train', 4: 'val', 5: 'test'}),
})
missing = esc['path'].isna().sum()
esc = esc.dropna(subset=['path'])
print(f'ESC-50: {len(esc)} clips ({missing} audio files not found)')

### FSD50K
200 classes of Freesound clips with human labels. v3 takes **every gunshot clip** and a capped sample of hard-negative
classes: impulsive household sounds and steady machine noise (rare classes first, at most 80 dev / 30 eval clips per
class), plus random other clips. FSD50K's own split is kept: dev `train` → train, dev `val` → validation,
**eval → test**. Dev and eval share no Freesound uploader.

In [ ]:
FSD_IMPULSIVE = [
    'Finger_snapping', 'Clapping', 'Slam', 'Knock', 'Door', 'Sliding_door', 'Drawer_open_or_close', 'Cupboard_open_or_close',
    'Dishes_and_pots_and_pans', 'Cutlery_and_silverware', 'Chink_and_clink', 'Glass', 'Shatter', 'Crack', 'Thump_and_thud',
    'Tap', 'Hammer', 'Wood', 'Tick', 'Coin_(dropping)', 'Keys_jangling', 'Computer_keyboard', 'Typing', 'Typewriter',
    'Camera', 'Scissors', 'Ratchet_and_pawl', 'Fireworks', 'Boom', 'Explosion', 'Thunder', 'Snare_drum', 'Bass_drum',
    'Drum', 'Hi-hat', 'Cymbal', 'Tabla', 'Writing', 'Packing_tape_and_duct_tape', 'Tearing', 'Crumpling_and_crinkling',
    'Crushing', 'Zipper_(clothing)', 'Walk_and_footsteps', 'Squeak', 'Mechanisms', 'Tools', 'Power_tool', 'Drill',
    'Sawing', 'Engine_starting', 'Cough', 'Sneeze', 'Splash_and_splatter']
FSD_MACHINE = [
    'Mechanical_fan', 'Microwave_oven', 'Printer', 'Engine', 'Idling', 'Accelerating_and_revving_and_vroom', 'Motorcycle',
    'Car_passing_by', 'Truck', 'Traffic_noise_and_roadway_noise', 'Buzz', 'Hiss', 'Boiling', 'Frying_(food)',
    'Water_tap_and_faucet', 'Toilet_flush', 'Rain', 'Wind', 'Rail_transport', 'Aircraft']
GUN = 'Gunshot_and_gunfire'

for f in ('dev.csv', 'eval.csv', 'vocabulary.csv'):
    fetch(f'{FSD_HF}/labels/{f}', f'{DATA}/fsd/{f}')
fetch(f'{FSD_HF}/metadata/eval_clips_info_FSD50K.json', f'{DATA}/fsd/eval_info.json')
vocab = set(pd.read_csv(f'{DATA}/fsd/vocabulary.csv', header=None)[1])
unknown = [c for c in FSD_IMPULSIVE + FSD_MACHINE + [GUN] if c not in vocab]
assert not unknown, f'not in the FSD50K vocabulary: {unknown}'


def fsd_select(df, part):
    df = df.assign(L=df['labels'].str.split(','))
    is_gun = df['L'].apply(lambda l: GUN in l)
    picked = [df[is_gun].assign(category=GUN, group='gunshot')]
    neg = df[~is_gun]
    taken = set()
    has = {c: neg['L'].apply(lambda l, c=c: c in l) for c in FSD_IMPULSIVE + FSD_MACHINE}
    for c in sorted(has, key=lambda c: has[c].sum()):          # rare classes first
        pool = neg[has[c] & ~neg.index.isin(taken)]
        pool = pool.sample(min(FSD_CAP[part], len(pool)), random_state=SEED)
        taken.update(pool.index)
        picked.append(pool.assign(category=c, group='impulsive' if c in FSD_IMPULSIVE else 'machine'))
    rest = neg[~neg.index.isin(taken)]
    picked.append(rest.sample(min(FSD_OTHER[part], len(rest)), random_state=SEED).assign(category='other', group='other'))
    out = pd.concat(picked)
    split = out['split'].values if part == 'dev' else np.full(len(out), 'test')
    return pd.DataFrame({
        'source': 'fsd50k', 'name': out['fname'].astype(str) + '.wav',
        'path': [f'{DATA}/fsd/{part}/{f}.wav' for f in out['fname']],
        'fsid': out['fname'].astype(int).values, 'label': (out['group'] == 'gunshot').astype(int).values,
        'category': out['category'].values, 'group': out['group'].values, 'fold': 0,
        'split': split, 'fsd_labels': out['labels'].values, 'fsd_part': part,
        'url': [f'{FSD_HF}/clips/{part}/{f}.wav' for f in out['fname']]})


fsd = pd.concat([fsd_select(pd.read_csv(f'{DATA}/fsd/dev.csv'), 'dev'),
                 fsd_select(pd.read_csv(f'{DATA}/fsd/eval.csv'), 'eval')], ignore_index=True)

# The three datasets all come from Freesound: drop any recording that is already in UrbanSound8K (or ESC-50)
n0 = len(esc), len(fsd)
esc = esc[~esc.fsid.isin(us8k.fsid)]
fsd = fsd[~fsd.fsid.isin(us8k.fsid) & ~fsd.fsid.isin(esc.fsid)]
print(f'duplicates removed: ESC-50 {n0[0] - len(esc)}, FSD50K {n0[1] - len(fsd)}')
print(fsd.groupby(['split', 'group']).size().unstack(fill_value=0))

In [ ]:
# Download the selected FSD50K clips (one file each)
def download(row):
    try:
        fetch(row.url, row.path)
        sf.info(row.path)
        return True
    except Exception as e:
        try:
            os.remove(row.path)
        except OSError:
            pass
        return repr(e)

t0 = time.time()
with ThreadPoolExecutor(16) as pool:
    results = list(pool.map(download, fsd.itertuples()))
ok = np.array([r is True for r in results])
errors = pd.Series([r for r in results if r is not True]).value_counts()
fsd = fsd[ok]
print(f'FSD50K: {ok.sum()} clips ready, {(~ok).sum()} failed — {time.time() - t0:.0f}s')
if len(errors):
    print(errors.head())

In [ ]:
clips = pd.concat([us8k, esc, fsd], ignore_index=True)
summary = clips.groupby(['source', 'split', 'label']).size().unstack(fill_value=0).rename(columns={0: 'other sounds', 1: 'gunshots'})
display(summary)
print(f"total: {len(clips)} clips, {clips.label.sum()} gunshots ({clips.label.mean():.1%})")

## 2. Windows, augmentation, features
* **Digital silence is trimmed.** ESC-50 pads short events with exact zeros, which only negatives would have.
  The model must not learn "perfect silence ⇒ not a gunshot".
* **One 4 s window per clip**: clips shorter than 4 s are loop-padded (as in v2); for longer clips the loudest
  4 s are used, where the labelled sound most likely is.
* **Augmentation** (training clips only, one extra copy each): room reverb, a cheap microphone's EQ, a background
  sound from ESC-50's training folds at 0–20 dB SNR, a noise floor, and soft clipping, as when a sound happens
  right next to the microphone (like the finger snap that fooled v2).

In [ ]:
SILENCE = 2e-5   # below one 16-bit step: only digital silence is trimmed
PINK_B = [0.049922035, -0.095993537, 0.050612699, -0.004408786]
PINK_A = [1, -2.494956002, 2.017265875, -0.522189400]


def trim_digital_silence(y):
    idx = np.flatnonzero(np.abs(y) > SILENCE)
    return y[idx[0]: idx[-1] + 1] if len(idx) else y[:0]


def loudest_window(y, n=N):
    if len(y) <= n:
        return F.loop_pad(y, n)
    c = np.concatenate([[0.0], np.cumsum(y.astype(np.float64) ** 2)])
    starts = np.arange(0, len(y) - n + 1, CFG.hop_length)
    s = starts[np.argmax(c[starts + n] - c[starts])]
    return y[s:s + n].astype(np.float32)


def rms(x):
    return float(np.sqrt(np.mean(np.square(x)))) + 1e-12


def augment(y, rng, bg=None):
    y = y.astype(np.float64)
    if rng.random() < 0.5:                                   # room reverb (synthetic impulse response)
        rt60 = rng.uniform(0.15, 0.8)
        t = np.arange(int(rt60 * SR)) / SR
        rir = rng.standard_normal(len(t)) * np.exp(-6.9 * t / rt60)
        wet = scipy.signal.fftconvolve(y, rir)[:len(y)]
        mix = rng.uniform(0.2, 0.7)
        y = (1 - mix) * y + mix * wet * (rms(y) / rms(wet))
    if rng.random() < 0.5:                                   # small microphone: weak bass
        sos = scipy.signal.butter(2, rng.uniform(80, 400), 'highpass', fs=SR, output='sos')
        y = scipy.signal.sosfilt(sos, y)
    if rng.random() < 0.3:                                   # ... or dull treble
        sos = scipy.signal.butter(2, rng.uniform(4000, 9000), 'lowpass', fs=SR, output='sos')
        y = scipy.signal.sosfilt(sos, y)
    if bg is not None:                                       # background sound, 0–20 dB below the clip
        y = y + bg * (rms(y) / rms(bg)) * 10 ** (-rng.uniform(0, 20) / 20)
    if rng.random() < 0.2:                                   # white or pink noise floor
        noise = rng.standard_normal(len(y))
        if rng.random() < 0.5:
            noise = scipy.signal.lfilter(PINK_B, PINK_A, noise)
        y = y + noise * (rms(y) / rms(noise)) * 10 ** (-rng.uniform(15, 40) / 20)
    if rng.random() < 0.2:                                   # overload: soft clipping
        g = rng.uniform(2, 10)
        y = np.tanh(g * y / (np.max(np.abs(y)) + 1e-12))
    return y.astype(np.float32)


def load_clip(path):
    return trim_digital_silence(F.load_audio(path, CFG))


def row_features(path, seed=-1, bg_path=None):
    # One clip -> (128, 173) float16 log-mel; seed >= 0 makes an augmented copy.
    try:
        y = loudest_window(load_clip(path))
        if seed >= 0:
            rng = np.random.default_rng(seed)
            bg = None
            if isinstance(bg_path, str):
                b = load_clip(bg_path)
                if len(b):
                    b = b[rng.integers(0, max(1, len(b) - N)):]
                    bg = F.loop_pad(b, N)
            y = augment(y, rng, bg)
        return F.log_mel(y, CFG).astype(np.float16)
    except Exception as e:
        print('skipped', os.path.basename(path), '-', e)
        return None

In [ ]:
rows = clips.assign(aug=False)
if AUGMENT:
    rows = pd.concat([rows, clips[clips.split.isin(['us8k', 'train'])].assign(aug=True)], ignore_index=True)
rows = rows.reset_index(drop=True)
rng = np.random.default_rng(SEED)
bank = esc[(esc.split == 'train') & esc.category.isin(BACKGROUND)]['path'].values
rows['seed'] = np.where(rows['aug'], np.arange(len(rows)), -1)
rows['bg'] = [bank[rng.integers(len(bank))] if a and len(bank) and rng.random() < 0.6 else None for a in rows['aug']]

CACHE = f'{DATA}/features_v3_{len(rows)}.npy'
t0 = time.time()
if os.path.exists(CACHE):
    X = np.load(CACHE)
    ok = np.load(CACHE.replace('.npy', '_ok.npy'))
    print('Loaded cached features')
else:
    feats = Parallel(n_jobs=-1, batch_size=16, verbose=2)(
        delayed(row_features)(p, s, b) for p, s, b in zip(rows['path'], rows['seed'], rows['bg']))
    ok = np.array([f is not None for f in feats])
    X = np.stack([f for f in feats if f is not None])
    del feats
    np.save(CACHE, X)
    np.save(CACHE.replace('.npy', '_ok.npy'), ok)
rows = rows[ok].reset_index(drop=True)
y = rows['label'].values.astype(np.float32)
print(f'X {X.shape} {X.dtype}, {X.nbytes / 1e9:.2f} GB, {int(y.sum())} gunshot rows — {time.time() - t0:.0f}s')

In [ ]:
# A few clean / augmented pairs, to check that augmentation looks sensible
pairs = []
for cat in ['gun_shot', 'Finger_snapping', 'Clapping', 'Mechanical_fan']:
    cand = rows[(rows.category == cat) & rows.aug]
    if len(cand):
        r = cand.iloc[0]
        clean = rows.index[(rows.path == r.path) & ~rows.aug][0]
        pairs.append((cat, clean, cand.index[0]))
if pairs:
    fig, axes = plt.subplots(2, len(pairs), figsize=(4.5 * len(pairs), 6), squeeze=False)
    for j, (cat, a, b) in enumerate(pairs):
        for i, (idx, kind) in enumerate([(a, 'original'), (b, 'augmented')]):
            axes[i, j].imshow(X[idx].astype(np.float32), origin='lower', aspect='auto', cmap='magma', vmin=0, vmax=1)
            axes[i, j].set_title(f'{cat} — {kind}'); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    plt.tight_layout(); plt.savefig(f'{ASSET_DIR}/augmentation.png', dpi=110); plt.show()

## 3. Model
The same network as v2 (so the browser demo needs no changes): three convolution blocks, global average + max
pooling, 304,289 parameters.

In [ ]:
def build_model(input_shape=(CFG.n_mels, CFG.n_frames, 1)):
    inp = keras.Input(shape=input_shape, name='log_mel')
    x = inp
    for filters in (32, 64, 128):
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.Concatenate()([layers.GlobalAveragePooling2D()(x), layers.GlobalMaxPooling2D()(x)])
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid', name='gunshot_probability')(x)
    model = keras.Model(inp, out, name='shotshield_v3')
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(name='auc'), keras.metrics.AUC(curve='PR', name='pr_auc'),
                           keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')])
    return model


def as_input(idx):
    return X[idx].astype(np.float32)[..., np.newaxis]


def train(train_idx, val_idx, verbose=0):
    keras.backend.clear_session()
    model = build_model()
    w = compute_class_weight('balanced', classes=np.array([0.0, 1.0]), y=y[train_idx])   # gunshots are rare
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max', factor=0.5, patience=3, min_lr=1e-5),
    ]
    history = model.fit(as_input(train_idx), y[train_idx], validation_data=(as_input(val_idx), y[val_idx]),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, class_weight={0: w[0], 1: w[1]},
                        callbacks=callbacks, verbose=verbose)
    return model, history


def predict(model, idx):
    idx = np.asarray(idx)
    if len(idx) == 0:
        return np.zeros(0)
    return np.concatenate([model.predict(as_input(c), batch_size=128, verbose=0).ravel()
                           for c in np.array_split(idx, int(np.ceil(len(idx) / 4096)))])


def binary_metrics(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, pred, average='binary', zero_division=0)
    return {'threshold': float(threshold), 'precision': float(p), 'recall': float(r), 'f1': float(f),
            'roc_auc': float(roc_auc_score(y_true, prob)), 'pr_auc': float(average_precision_score(y_true, prob)),
            'accuracy': float((pred == y_true).mean())}


# Row masks
SRC, SPLIT, FOLD, AUG = rows['source'].values, rows['split'].values, rows['fold'].values, rows['aug'].values
EXTRA_TRAIN = SPLIT == 'train'                       # ESC-50 folds 1–3, FSD50K dev-train (+ augmented copies)
EXTRA_VAL = (SPLIT == 'val') & ~AUG                  # ESC-50 fold 4, FSD50K dev-val
HARD_TEST = np.where((SPLIT == 'test') & ~AUG)[0]    # ESC-50 fold 5, FSD50K eval: never trained on


def us8k_mask(folds, augmented=False):
    m = (SRC == 'us8k') & np.isin(FOLD, folds)
    return m if augmented else m & ~AUG


def hard_report(prob, threshold=0.5):
    neg = y[HARD_TEST] == 0
    return {'hard_neg_false_alarm_rate': float((prob[neg] >= threshold).mean()),
            'fsd_gunshot_recall': float((prob[~neg] >= threshold).mean()) if (~neg).any() else float('nan')}


print(f'hard test set: {int((y[HARD_TEST] == 0).sum())} other sounds, {int(y[HARD_TEST].sum())} FSD50K gunshots')

## 4. 10-fold cross-validation on UrbanSound8K
Same protocol as v2 (test fold k, validation fold k+1, the other eight train), so the numbers are directly
comparable. ESC-50/FSD50K training clips are added to every training set and their validation clips to every
validation set. Each fold's model is also scored on the hard test set.

In [ ]:
oof = np.full(len(rows), np.nan)
cv_rows, hard_cv = [], []
if RUN_CV:
    for k in range(1, 11):
        val_k = k % 10 + 1
        test_idx = np.where(us8k_mask([k]))[0]
        val_idx = np.where(us8k_mask([val_k]) | EXTRA_VAL)[0]
        train_idx = np.where(us8k_mask([f for f in range(1, 11) if f not in (k, val_k)], augmented=True) | EXTRA_TRAIN)[0]
        t0 = time.time()
        model, history = train(train_idx, val_idx)
        oof[test_idx] = predict(model, test_idx)
        hard_prob = predict(model, HARD_TEST)
        hard_cv.append(hard_prob)
        m = binary_metrics(y[test_idx], oof[test_idx])
        m.update(hard_report(hard_prob))
        m.update(fold=k, epochs=len(history.history['loss']), minutes=round((time.time() - t0) / 60, 1))
        cv_rows.append(m)
        print(f"fold {k:2d} | precision {m['precision']:.3f} recall {m['recall']:.3f} F1 {m['f1']:.3f} "
              f"ROC-AUC {m['roc_auc']:.3f} | hard negatives: {m['hard_neg_false_alarm_rate']:.1%} false alarms, "
              f"FSD50K gunshots {m['fsd_gunshot_recall']:.1%} caught | {m['epochs']} epochs, {m['minutes']} min")
        del model; gc.collect()

    cv_df = pd.DataFrame(cv_rows).set_index('fold')
    cv_df.to_csv(f'{OUT_DIR}/cv_results_v3.csv')
    cols = ['precision', 'recall', 'f1', 'roc_auc', 'pr_auc', 'hard_neg_false_alarm_rate', 'fsd_gunshot_recall']
    cv_summary = cv_df[cols].agg(['mean', 'std'])
    v2_cv = pd.read_csv(fetch(f'{REPO_RAW}/results/cv_results.csv', f'{DATA}/v2_cv_results.csv'))
    compare = pd.DataFrame({
        'v2 (UrbanSound8K only)': [f'{v2_cv[c].mean():.3f} ± {v2_cv[c].std():.3f}' for c in cols[:5]] + ['–', '–'],
        'v3 (+ hard negatives)': [f"{cv_summary.loc['mean', c]:.3f} ± {cv_summary.loc['std', c]:.3f}" for c in cols]},
        index=cols)
    display(compare)

    us_oof = rows.index[(SRC == 'us8k') & ~AUG]
    fa = rows.loc[us_oof][(y[us_oof] == 0) & (oof[us_oof] >= 0.5)]['category'].value_counts()
    print(f'UrbanSound8K false alarms (threshold 0.5): {fa.sum()} (v2: 52)'); print(fa.to_string())
    print(f'Missed gunshots: {int(((y[us_oof] == 1) & (oof[us_oof] < 0.5)).sum())} of {int(y[us_oof].sum())} (v2: 42)')
else:
    print('RUN_CV = False — skipping cross-validation')

## 5. Final model
Trained on UrbanSound8K folds 1–8 + ESC-50/FSD50K training clips; early-stopped and threshold-tuned on fold 9 +
ESC-50/FSD50K validation clips; tested on fold 10 and the hard test set. The released **v2** model is scored on the
same clips.

In [ ]:
train_idx = np.where(us8k_mask(range(1, 9), augmented=True) | EXTRA_TRAIN)[0]
val_idx = np.where(us8k_mask([9]) | EXTRA_VAL)[0]
test_idx = np.where(us8k_mask([10]))[0]

final_model, history = train(train_idx, val_idx, verbose=2)

val_prob = predict(final_model, val_idx)
p, r, t = precision_recall_curve(y[val_idx], val_prob)
f1 = 2 * p[:-1] * r[:-1] / np.maximum(p[:-1] + r[:-1], 1e-9)
THRESHOLD = float(t[np.argmax(f1)])
print(f'Threshold chosen on the validation set: {THRESHOLD:.3f}')

test_prob = predict(final_model, test_idx)
hard_prob = predict(final_model, HARD_TEST)

# The released v2 model, on exactly the same inputs
v2_info = json.load(open(fetch(f'{V2_RELEASE}/model_info.json', f'{DATA}/v2/model_info.json')))
v2_path = fetch(f'{V2_RELEASE}/shotshield_v2.keras', f'{DATA}/v2/shotshield_v2.keras')
assert sha256(v2_path) == v2_info['model_sha256'], 'v2 model checksum mismatch'
v2_model = keras.models.load_model(v2_path)
v2_test = predict(v2_model, test_idx)
v2_hard = predict(v2_model, HARD_TEST)
v2_val = predict(v2_model, val_idx)
V2_THRESHOLD = v2_info['threshold']


def summary_row(name, thr, p_test, p_hard):
    m = binary_metrics(y[test_idx], p_test, thr)
    neg = y[HARD_TEST] == 0
    return {'model': name, 'threshold': round(thr, 3),
            'US8K fold 10 precision': m['precision'], 'recall': m['recall'], 'F1': m['f1'],
            'hard negatives: false alarms': f'{int((p_hard[neg] >= thr).sum())}/{int(neg.sum())} ({(p_hard[neg] >= thr).mean():.1%})',
            'FSD50K gunshots caught': f'{int((p_hard[~neg] >= thr).sum())}/{int((~neg).sum())} ({(p_hard[~neg] >= thr).mean():.1%})'}


final_table = pd.DataFrame([summary_row('v2', V2_THRESHOLD, v2_test, v2_hard),
                            summary_row('v3', THRESHOLD, test_prob, hard_prob)]).set_index('model')
display(final_table.round(3))

# The threshold trades false alarms against missed gunshots: v3 at a few settings
neg_h, pos_h = y[HARD_TEST] == 0, y[HARD_TEST] == 1
sweep = pd.DataFrame([{'threshold': t,
                       'US8K fold 10 recall': float((test_prob[y[test_idx] == 1] >= t).mean()),
                       'US8K fold 10 false alarms': int((test_prob[y[test_idx] == 0] >= t).sum()),
                       'hard-negative false-alarm rate': float((hard_prob[neg_h] >= t).mean()),
                       'FSD50K gunshot recall': float((hard_prob[pos_h] >= t).mean())}
                      for t in sorted({0.25, 0.5, 0.7, 0.8, 0.9, 0.95, round(THRESHOLD, 3)})]).set_index('threshold')
display(sweep.round(3))

In [ ]:
# False alarms on the hard test set, by kind of sound (each model at its own threshold)
hard = rows.loc[HARD_TEST, ['source', 'category', 'group', 'label']].assign(v2=v2_hard >= V2_THRESHOLD, v3=hard_prob >= THRESHOLD)
neg = hard[hard.label == 0]
by_group = neg.groupby(['source', 'group'])[['v2', 'v3']].mean().mul(100).round(1)
by_group['clips'] = neg.groupby(['source', 'group']).size()
display(by_group)

KEY = ['Finger_snapping', 'Clapping', 'Knock', 'Slam', 'Door', 'Dishes_and_pots_and_pans', 'Typing', 'Computer_keyboard',
       'Fireworks', 'Explosion', 'Mechanical_fan', 'Microwave_oven', 'Engine', 'Motorcycle',
       'clapping', 'door_wood_knock', 'mouse_click', 'keyboard_typing', 'can_opening', 'glass_breaking', 'fireworks',
       'vacuum_cleaner', 'washing_machine']
by_class = neg.groupby('category')[['v2', 'v3']].agg(['sum', 'count'])
by_class = pd.DataFrame({'clips': by_class[('v2', 'count')], 'v2 false alarms': by_class[('v2', 'sum')],
                         'v3 false alarms': by_class[('v3', 'sum')]})
by_class.to_csv(f'{OUT_DIR}/hard_test_false_alarms_by_class.csv')
display(by_class.loc[[c for c in KEY if c in by_class.index]])
print('Classes v3 still confuses most:')
display(by_class.sort_values('v3 false alarms', ascending=False).head(10))

fig, ax = plt.subplots(figsize=(9, 0.45 * len(by_group) + 1.5))
labels = [f'{s} · {g}' for s, g in by_group.index]
yy = np.arange(len(labels))
ax.barh(yy - 0.2, by_group['v2'], 0.4, label=f'v2 (threshold {V2_THRESHOLD:.2f})', color='#9aa0a6')
ax.barh(yy + 0.2, by_group['v3'], 0.4, label=f'v3 (threshold {THRESHOLD:.2f})', color='#d62728')
ax.set_yticks(yy, labels); ax.invert_yaxis(); ax.set_xlabel('false-alarm rate (%)')
ax.set_title('False alarms on sounds no model was trained on'); ax.legend()
plt.tight_layout(); plt.savefig(f'{ASSET_DIR}/v2_vs_v3_false_alarms.png', dpi=130); plt.show()

## 6. Save and package

In [ ]:
MODEL_PATH = f'{OUT_DIR}/shotshield_v3.keras'
final_model.save(MODEL_PATH)
reloaded = keras.models.load_model(MODEL_PATH)
assert np.allclose(predict(reloaded, test_idx[:32]), test_prob[:32], atol=1e-5), 'reload mismatch'

h = history.history
neg = y[HARD_TEST] == 0
info = {
    'name': 'ShotShield', 'version': '3.0',
    'created': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'task': 'binary: does a 4 s audio window contain a gunshot?',
    'labels': {'0': 'not_gunshot', '1': 'gun_shot'},
    'threshold': THRESHOLD,
    'features': {'sample_rate': SR, 'clip_seconds': CFG.clip_seconds, 'n_mels': CFG.n_mels, 'n_fft': CFG.n_fft,
                 'hop_length': CFG.hop_length, 'top_db': CFG.top_db, 'n_frames': CFG.n_frames, 'padding': 'loop'},
    'input_shape': [CFG.n_mels, CFG.n_frames, 1],
    'parameters': int(final_model.count_params()),
    'training': {
        'datasets': {'UrbanSound8K': 'folds 1-8 train, 9 validation, 10 test',
                     'ESC-50': 'folds 1-3 train, 4 validation, 5 test',
                     'FSD50K': 'dev train / dev val / eval test; all gunshot clips + capped hard-negative classes'},
        'clips': {f'{s}/{sp}': {'other': int(((clips.source == s) & (clips.split == sp) & (clips.label == 0)).sum()),
                                'gunshot': int(((clips.source == s) & (clips.split == sp) & (clips.label == 1)).sum())}
                  for s, sp in clips.groupby(['source', 'split']).size().index},
        'deduplication': 'ESC-50/FSD50K clips whose Freesound ID is in UrbanSound8K removed; FSD50K vs ESC-50 too',
        'windows': 'digital silence trimmed; loop-padded if < 4 s, else the loudest 4 s',
        'augmentation': 'one copy per training clip: reverb, mic EQ, ESC-50 background 0-20 dB SNR, noise, soft clipping' if AUGMENT else None,
        'fsd_selection': {'cap': FSD_CAP, 'other': FSD_OTHER, 'impulsive': FSD_IMPULSIVE, 'machine': FSD_MACHINE},
        'epochs_run': len(h['loss']), 'class_weighting': 'balanced'},
    'test_fold_10': {'threshold_0.5': binary_metrics(y[test_idx], test_prob, 0.5),
                     'tuned_threshold': binary_metrics(y[test_idx], test_prob, THRESHOLD)},
    'hard_test': {'description': 'ESC-50 fold 5 + FSD50K eval (never trained on)',
                  'other_sounds': int(neg.sum()), 'gunshots': int((~neg).sum()),
                  'v3': hard_report(hard_prob, THRESHOLD), 'v2': hard_report(v2_hard, V2_THRESHOLD)},
    'cross_validation': ({k: {'mean': float(cv_summary.loc['mean', k]), 'std': float(cv_summary.loc['std', k])}
                          for k in cv_summary.columns} if RUN_CV else None),
    'versions': {'tensorflow': tf.__version__, 'keras': keras.__version__, 'librosa': librosa.__version__},
    'model_sha256': sha256(MODEL_PATH),
}
json.dump(info, open(f'{OUT_DIR}/model_info.json', 'w'), indent=2)
clips[['source', 'name', 'fsid', 'label', 'category', 'group', 'fold', 'split']].to_csv(f'{OUT_DIR}/data_manifest.csv', index=False)
pcols = ['source', 'name', 'fsid', 'category', 'group', 'label']
pd.concat([rows.loc[test_idx, pcols].assign(set='us8k_fold10', v2=v2_test, v3=test_prob),
           rows.loc[HARD_TEST, pcols].assign(set='hard_test', v2=v2_hard, v3=hard_prob),
           rows.loc[val_idx, pcols].assign(set='validation', v2=v2_val, v3=val_prob)]
          ).to_csv(f'{OUT_DIR}/predictions.csv', index=False)   # per-clip probabilities, for choosing a threshold later
sweep.to_csv(f'{OUT_DIR}/threshold_sweep.csv')
print(json.dumps(info['hard_test'], indent=1))

## 7. Browser bundle
Converts v3 to ONNX (tf2onnx, which worked for v2), checks it against Keras with onnxruntime, and packages example
clips: the same UrbanSound8K test-fold picks as v2, plus FSD50K **test** clips of the sounds that fooled v2
(finger snaps, claps, knocks, …), with their Freesound credits. `tools/install_web_bundle.py` installs it.

In [ ]:
CONVERT = r'''
import os, sys
import numpy as np
strategy, model_path, out_path = sys.argv[1:4]
import keras
model = keras.models.load_model(model_path)
model(np.zeros((1,) + tuple(model.inputs[0].shape[1:]), 'float32'))   # export needs a model that has been called
if strategy == 'tf2onnx':
    import tensorflow as tf, tf2onnx
    spec = (tf.TensorSpec((None,) + tuple(model.inputs[0].shape[1:]), tf.float32, name='log_mel'),)
    fn = tf.function(lambda x: model(x, training=False), input_signature=spec)
    tf2onnx.convert.from_function(fn, input_signature=spec, opset=17, output_path=out_path)
else:
    model.export(out_path, format='onnx')
'''

# Example clips
examples, waves = [], {}
fold10 = clips[(clips.source == 'us8k') & (clips.fold == 10)]
gun = fold10['category'] == 'gun_shot'
picks = [fold10[gun].sample(2, random_state=7)]
for cls in ['dog_bark', 'siren', 'drilling', 'jackhammer', 'air_conditioner', 'car_horn']:
    picks.append(fold10[fold10['category'] == cls].sample(1, random_state=7))
lic_ok = ('publicdomain/zero', 'licenses/by/', 'licenses/by-nc/')
fsd_info = json.load(open(f'{DATA}/fsd/eval_info.json'))
fsd_test = clips[(clips.source == 'fsd50k') & (clips.split == 'test') & (clips.label == 0)]
for cls in ['Finger_snapping', 'Clapping', 'Knock', 'Slam', 'Dishes_and_pots_and_pans', 'Typing', 'Fireworks', 'Mechanical_fan']:
    cand = fsd_test[(fsd_test.category == cls) & fsd_test.fsid.astype(str).map(
        lambda f: any(k in fsd_info.get(f, {}).get('license', '') for k in lic_ok))]
    cand = cand[[sf.info(p).duration <= 8 for p in cand.path]]
    if len(cand):
        picks.append(cand.sample(1, random_state=7))
chosen = pd.concat(picks)

for _, r in chosen.iterrows():
    stem = os.path.splitext(r['name'])[0]
    name = f"{r['category']}_{stem}".replace('(', '').replace(')', '')
    y_ex = F.load_audio(r['path'], CFG)
    wav = f'{WEB_DIR}/examples/{name}.wav'
    sf.write(wav, np.clip(np.round(y_ex * 32767), -32768, 32767).astype(np.int16), SR, subtype='PCM_16')
    waves[name], _ = sf.read(wav, dtype='float32')      # what the browser decodes
    e = {'name': name, 'file': f'examples/{name}.wav', 'class': r['category'].lower(),
         'is_gunshot': bool(r['label']), 'duration': round(len(waves[name]) / SR, 3), 'dataset': r['source'],
         'slice_file_name': r['name'], 'fsID': int(r['fsid']), 'source': f"https://freesound.org/s/{int(r['fsid'])}/",
         'fold': 10 if r['source'] == 'us8k' else None, 'note': None}
    if r['source'] == 'fsd50k':
        meta_ = fsd_info.get(str(int(r['fsid'])), {})
        e.update(note='FSD50K eval set (never trained on)',
                 credit={'title': meta_.get('title'), 'uploader': meta_.get('uploader'), 'license': meta_.get('license')})
    examples.append(e)

# Reference data: same synthetic signals as the v2 export, plus the example clips
rng_ref = np.random.default_rng(2026)
t_ref = np.arange(N) / SR
shots = np.zeros(N)
for s in (0.4, 1.6, 1.75, 3.1):
    i = int(s * SR); shots[i:i + SR // 2] += 0.9 * rng_ref.standard_normal(SR // 2) * np.exp(-np.arange(SR // 2) / (0.03 * SR))
waves.update({'synth_white_noise': (0.3 * rng_ref.standard_normal(N)).astype(np.float32),
              'synth_impulses_in_noise': (shots + 0.01 * rng_ref.standard_normal(N)).astype(np.float32),
              'synth_chirp': (0.5 * np.sin(2 * np.pi * 50 * 4 / np.log(200) * (200 ** (t_ref / 4) - 1))).astype(np.float32)})
arrays, feats_all, probs_all = {}, [], []
for name, wv in waves.items():
    fe, st = F.windows_to_features(wv, CFG)
    pr = final_model.predict(fe, verbose=0).ravel()
    arrays.update({f'wave/{name}': wv, f'feat/{name}': fe, f'starts/{name}': np.asarray(st), f'prob/{name}': pr.astype(np.float32)})
    feats_all.append(fe); probs_all.append(pr)
for e in examples:
    pr = arrays[f"prob/{e['name']}"]
    e.update(keras_window_probs=[round(float(v), 6) for v in pr], keras_max_prob=round(float(pr.max()), 6),
             keras_detected=bool(pr.max() >= THRESHOLD))
np.savez_compressed(f'{WEB_DIR}/reference.npz', info=json.dumps({'librosa': librosa.__version__, 'keras': keras.__version__,
                    'threshold': THRESHOLD, 'model': 'v3'}), **arrays)
X_ref, p_keras = np.concatenate(feats_all).astype(np.float32), np.concatenate(probs_all)

# ONNX
import onnx, onnxruntime as ort
chosen_onnx = None
for strategy in ['tf2onnx', 'keras_tf']:
    path = f'{DATA}/v3_{strategy}.onnx'
    r = subprocess.run([sys.executable, '-c', CONVERT, strategy, MODEL_PATH, path], capture_output=True, text=True, timeout=1200)
    if r.returncode or not os.path.exists(path):
        print(strategy, 'failed:', (r.stderr or r.stdout).strip().splitlines()[-1:])
        continue
    m = onnx.load(path)
    sess = ort.InferenceSession(m.SerializeToString(), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0]
    p_onnx = np.concatenate([sess.run(None, {inp.name: X_ref[i:i + 1]})[0].ravel() for i in range(len(X_ref))])
    err = float(np.abs(p_onnx - p_keras).max())
    print(f'{strategy}: max |ONNX − Keras| = {err:.2e} over {len(X_ref)} windows')
    if err < 1e-4:
        chosen_onnx = (strategy, m, inp, err)
        break
assert chosen_onnx, 'ONNX conversion failed — send the output above'
strategy, onnx_model, inp, err = chosen_onnx
onnx.save(onnx_model, f'{WEB_DIR}/shotshield_v3.onnx')

web_info = dict(info)
web_info['web'] = {'onnx_file': 'shotshield_v3.onnx', 'onnx_sha256': sha256(f'{WEB_DIR}/shotshield_v3.onnx'),
                   'converter': strategy, 'input_name': inp.name, 'input_shape': str(inp.shape),
                   'opset': max(o.version for o in onnx_model.opset_import if o.domain in ('', 'ai.onnx')),
                   'max_abs_diff_vs_keras': err, 'checked_windows': int(len(X_ref)),
                   'versions': {'onnx': onnx.__version__, 'onnxruntime': ort.__version__}}
json.dump(web_info, open(f'{WEB_DIR}/model_info.json', 'w'), indent=2)
json.dump({'description': 'UrbanSound8K test fold 10 (CC BY-NC 3.0) and FSD50K eval set (per-clip Creative Commons '
                          'licenses); clips originally from freesound.org',
           'threshold': THRESHOLD, 'examples': examples}, open(f'{WEB_DIR}/examples/examples.json', 'w'), indent=2)
display(pd.DataFrame(examples)[['name', 'is_gunshot', 'duration', 'keras_max_prob', 'keras_detected']])

for d in (OUT_DIR, WEB_DIR):
    z = shutil.make_archive(d + '_outputs' if d == OUT_DIR else d, 'zip', d)
    print(f'{z}: {os.path.getsize(z) / 1e6:.1f} MB, sha256 {sha256(z)}')

## 8. Results for the README

In [ ]:
def as_markdown(df):
    try:
        return df.to_markdown()
    except ImportError:          # tabulate missing
        return df.to_string()


print(as_markdown(final_table))
if RUN_CV:
    print(); print(as_markdown(compare))
print(f"\nParameters: {final_model.count_params():,} | threshold: {THRESHOLD:.3f}")

In [ ]:
# Stop the background worker processes so the run can end cleanly
from joblib.externals.loky import get_reusable_executor
get_reusable_executor().shutdown(wait=True)
print('DONE — download shotshield_v3_outputs.zip and shotshield_v3_web.zip from /kaggle/working')